In [ ]:
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "neurocnl",
#     "snntorch",
# ]
# ///

# NeuroMorphic Pipeline — Snntorch Sim

Generated 2026-07-04 00:00 UTC.

**Architecture:** defined in the Architecture tab (CNL spec below).
**Pipeline config:** edit `config` in the next cell to change training parameters.

In [ ]:
# ── Pipeline configuration ───────────────────────────────────────────
# Workspace settings used when this notebook was generated.

config = {
    "dataset":   "paper/03_rnn/data/ds_test.pt",
    "framework": "snntorch_sim",
}

print('Config loaded:', config)

In [ ]:
import json as _json


def _nmtk_emit(
    epoch: int, total: int, loss: float, accuracy: float, layer_rates: dict
) -> None:
    print(
        _json.dumps(
            {
                "__nmtk_progress__": True,
                "epoch": epoch,
                "total_epochs": total,
                "loss": loss,
                "accuracy": accuracy,
                "layer_spike_rates": layer_rates,
            }
        ),
        flush=True,
    )


In [ ]:
import torch
from torch.utils.data import DataLoader
# Load your custom dataset from: paper/03_rnn/data/ds_test.pt
# train_ds = torch.load('paper/03_rnn/data/ds_test.pt/train.pt')
# test_ds  = torch.load('paper/03_rnn/data/ds_test.pt/test.pt')
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)


## Architecture

Network compiled from CNL spec via NIR.

In [ ]:
# CNL spec — auto-generated from the Architecture canvas tab.
# To change the network, edit the Architecture tab and regenerate.
cnl_spec = '''
Define a network named graph.
# Network with 1 input, 5 hidden nodes, 1 output.
# flow: input → fc1 → lif1.lif → lif2 → output → lif1.lif → lif1.w_rec → lif1.lif → fc2 (recurrent: lif1.w_rec ↺ lif1.lif, fc2 ↺ lif2)

# Layers:
Define an affine transformation named fc1 with weight matrix shape (38, 12) and bias vector shape (38,).
Define an affine transformation named fc2 with weight matrix shape (7, 38) and bias vector shape (7,).
Define an input port named input with shape (12,).
Define a current-based leaky integrate-and-fire neuron named lif1.lif with synaptic time constant shape (38,), membrane time constant shape (38,), resistance shape (38,), leak voltage shape (38,), firing threshold shape (38,), and input weight shape (38,).
Define an affine transformation named lif1.w_rec with weight matrix shape (38, 38) and bias vector shape (38,).
Define a current-based leaky integrate-and-fire neuron named lif2 with synaptic time constant shape (7,), membrane time constant shape (7,), resistance shape (7,), leak voltage shape (7,), firing threshold shape (7,), and input weight shape (7,).
Define an output port named output with shape (7,).

# Connections:
input connects to fc1.
fc1 connects to lif1.lif.
lif1.w_rec connects to lif1.lif.
lif2 connects to output.
fc2 connects to lif2.
lif1.lif connects to lif1.w_rec.
lif1.lif connects to fc2.

'''

from neurocnl.compile import compile_to_nir

graph = compile_to_nir(cnl_spec)
print(f'Network: {len(graph.nodes)} nodes, {len(graph.edges)} edges')

In [ ]:
"""snnTorch network — auto-generated from NIR graph."""

import torch
import torch.nn as nn
import snntorch as snn
import numpy as np

_w = np.load('weights.npz')  # weights file saved alongside this notebook


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Linear layer: 'fc1'  shape (38, 12)
        self.fc1 = nn.Linear(12, 38, bias=True)
        self.fc1.weight.data = torch.from_numpy(_w['fc1_weight'].copy())
        self.fc1.bias.data = torch.from_numpy(_w['fc1_bias'].copy())
        # Linear layer: 'fc2'  shape (7, 38)
        self.fc2 = nn.Linear(38, 7, bias=True)
        self.fc2.weight.data = torch.from_numpy(_w['fc2_weight'].copy())
        self.fc2.bias.data = torch.from_numpy(_w['fc2_bias'].copy())
        # LIF population: 'lif1.lif'  (1 neurons)
        self.lif1_lif = snn.Leaky(beta=0.950000, threshold=1.0000, init_hidden=True)
        # ponytail: 'lif1.lif' approximates CubaLIF with snn.Leaky; synaptic current filtering is not modelled.
        # Linear layer: 'lif1.w_rec'  shape (38, 38)
        self.lif1_w_rec = nn.Linear(38, 38, bias=True)
        self.lif1_w_rec.weight.data = torch.from_numpy(_w['lif1_w_rec_weight'].copy())
        self.lif1_w_rec.bias.data = torch.from_numpy(_w['lif1_w_rec_bias'].copy())
        # LIF population: 'lif2'  (1 neurons)
        self.lif2 = snn.Leaky(beta=0.950000, threshold=1.0000, init_hidden=True)
        # ponytail: 'lif2' approximates CubaLIF with snn.Leaky; synaptic current filtering is not modelled.

    def forward(self, x):
        # initialise hidden states
        self.lif1_lif.init_leaky()
        self.lif2.init_leaky()
        # x: (T,B,C,H,W) time-first from tonic, or (B,C,H,W) for a single frame
        if x.dim() == 4:
            x = x.unsqueeze(0)  # (B,C,H,W) → (1,B,C,H,W)
        _x_seq = x
        spk_rec = []
        for t in range(_x_seq.shape[0]):
            x = _x_seq[t]
            x = self.lif1_w_rec(x)
            spk_lif2 = self.lif2(x)
            x = spk_lif2
            x = self.fc2(x)
            spk_lif1_lif = self.lif1_lif(x)
            x = spk_lif1_lif
            x = self.fc1(x)
            spk_lif1_lif = self.lif1_lif(x)
            x = spk_lif1_lif
            spk_rec.append(x)
        return torch.stack(spk_rec, dim=0), x  # (T, batch, out), last spk


net = Net().float()  # ponytail: npz weights load as float64; cast to match DataLoader float32 input
print(f'Net: {sum(p.numel() for p in net.parameters())} parameters')

> **⚠️ APPROXIMATE support for `snntorch_sim`**
>
> snnTorch simulator: nir.CubaLIF is executed with approximate semantics (timestep quantization may differ). Results are labelled 'approximate'.

In [ ]:
# ── Download as Python script ──────────────────────────────────
# Run this cell to download the notebook as a .py script.
import subprocess
subprocess.run(['jupyter', 'nbconvert', '--to', 'script',
                '__file__'], check=False)
print('Conversion triggered — check the file listing.')